<a href="https://colab.research.google.com/github/praveenvadde1250/fittrack-project-2601018/blob/main/Pandas_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Task 1: Load and Inspect the Data
Create the DataFrame using the provided code, then:

Display basic information using df.info()
Count missing values in each column using df.isna().sum()

In [1]:
import pandas as pd
import numpy as np

data = {
    'transaction_id': range(1, 21),
    'date': pd.date_range('2024-10-01', periods=20, freq='D'),
    'region': ['North', 'South', 'East', 'West', None, 'North', 'South', None, 'East', 'West',
               'North', 'South', 'East', 'West', 'North', None, 'East', 'West', 'North', 'South'],
    'product_category': ['Electronics', 'Clothing', None, 'Books', 'Electronics', 'Home',
                         'Clothing', 'Books', 'Electronics', None, 'Home', 'Clothing',
                         'Books', 'Electronics', 'Home', 'Clothing', 'Books', 'Electronics',
                         'Home', 'Clothing'],
    'sales_amount': [1200, 450, 890, None, 1500, 670, None, 340, 2100, 780,
                     560, None, 420, 1800, 920, 510, 380, None, 1100, 640],
    'quantity': [2, 5, 3, 1, None, 4, 2, 3, 1, 5,
                 3, 2, None, 1, 4, 3, 2, 1, None, 4],
    'customer_age': [25, 34, None, 45, 29, None, 38, 52, 27, 41,
                     33, None, 48, 26, 35, 42, None, 31, 39, 44],
    'payment_method': ['Credit Card', 'UPI', 'Cash', 'Debit Card', 'Credit Card',
                       'UPI', 'Cash', None, 'Credit Card', 'UPI', 'Debit Card',
                       'Cash', 'Credit Card', None, 'UPI', 'Cash', 'Debit Card',
                       'Credit Card', 'UPI', None]
}

df = pd.DataFrame(data)


Task 2: Handle Missing Values
Clean the dataset using appropriate strategies:

Region & Product_category: Fill with mode (most frequent value)
Sales_amount: Fill with median
Quantity: Fill using forward fill (ffill)
Customer_age: Fill with mean (rounded to integer)
Payment_method: Drop rows where payment_method is missing
Verify no missing values remain using df.isna().sum()

In [3]:
# Fill categorical columns with mode
df['region'].fillna(df['region'].mode()[0], inplace=True)
df['product_category'].fillna(df['product_category'].mode()[0], inplace=True)

# Fill numeric columns
df['sales_amount'].fillna(df['sales_amount'].median(), inplace=True)
df['quantity'].fillna(method='ffill', inplace=True)

# Fill age with rounded mean
df['customer_age'].fillna(round(df['customer_age'].mean()), inplace=True)

# Drop rows with missing payment method
df.dropna(subset=['payment_method'], inplace=True)

# Verify
print(df.isna().sum())

transaction_id      0
date                0
region              0
product_category    0
sales_amount        0
quantity            0
customer_age        0
payment_method      0
dtype: int64


/tmp/ipykernel_239/1296285257.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['region'].fillna(df['region'].mode()[0], inplace=True)
/tmp/ipykernel_239/1296285257.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)

Task 3: GroupBy Analysis
Perform the following aggregations:

Calculate total sales by region
Calculate average sales by product_category
Group by both region and product_category, calculate total sales and quantity, then use reset_index() to flatten
Display top 3 region-product combinations by sales

In [6]:
# Total sales by region
total_sales_by_region = df.groupby('region')['sales_amount'].sum()

# Average sales by product category
avg_sales_by_category = df.groupby('product_category')['sales_amount'].mean()

# Region + Product category summary
region_product_summary = df.groupby(
    ['region', 'product_category']
)[['sales_amount','quantity']].sum().reset_index()

# Top 3 combinations by sales
top3_sales = region_product_summary.sort_values(
    by='sales_amount', ascending=False
).head(3)

print(total_sales_by_region)
print(avg_sales_by_category)
print(region_product_summary)
print(top3_sales)

region
East     3790.0
North    6460.0
South    1900.0
West     2230.0
Name: sales_amount, dtype: float64
product_category
Books           508.333333
Clothing        680.000000
Electronics    1381.250000
Home            812.500000
Name: sales_amount, dtype: float64
  region product_category  sales_amount  quantity
0   East            Books         800.0       4.0
1   East         Clothing         890.0       3.0
2   East      Electronics        2100.0       1.0
3  North         Clothing         510.0       3.0
4  North      Electronics        2700.0       3.0
5  North             Home        3250.0      12.0
6  South         Clothing        1900.0       9.0
7   West            Books         725.0       1.0
8   West         Clothing         780.0       5.0
9   West      Electronics         725.0       1.0
  region product_category  sales_amount  quantity
5  North             Home        3250.0      12.0
4  North      Electronics        2700.0       3.0
2   East      Electronics        2

Task 4: Custom Aggregation
Create a function sales_range() that returns max - min of sales
Apply it to find sales range for each region
Use .agg() to calculate multiple metrics by region:
sales_amount: sum, mean, max
quantity: sum, min




In [8]:
# Custom function
def sales_range(x):
    return x.max() - x.min()

# Sales range by region
sales_range_by_region = df.groupby('region')['sales_amount'].agg(sales_range)

# Multiple aggregations
region_metrics = df.groupby('region').agg({
    'sales_amount': ['sum','mean','max'],
    'quantity': ['sum','min']
})

# Flatten columns
region_metrics.columns = ['_'.join(col) for col in region_metrics.columns]
region_metrics = region_metrics.reset_index()

print(sales_range_by_region)
print(region_metrics)

region
East     1720.0
North     990.0
South     275.0
West       55.0
Name: sales_amount, dtype: float64
  region  sales_amount_sum  sales_amount_mean  sales_amount_max  quantity_sum  \
0   East            3790.0         947.500000            2100.0           8.0   
1  North            6460.0         922.857143            1500.0          18.0   
2  South            1900.0         633.333333             725.0           9.0   
3   West            2230.0         743.333333             780.0           7.0   

   quantity_min  
0           1.0  
1           1.0  
2           2.0  
3           1.0  
